# Django ORM

## Settings

In [121]:
# gestion asynchronous mode
import os
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = 'true'
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'netpy.settings')

# other imports

# Python
from datetime import date

# Django
from django.db import connection, reset_queries
from django.db.models import Count, Prefetch, F, Q, Value, TextField
from django.db.models.functions import Extract, Coalesce


# Pour executer sans shell_plus --lab
import django
django.setup()

# Apps
from app_people.models import Person
from app_movie.models import Movie, Play

## Cleanup

In [46]:
Play.objects.all().delete()
Movie.objects.all().delete()
Person.objects.all().delete()

(115, {'app_people.Person': 115})

## CRUD

In [47]:
daniel_craig = Person.objects.create(name="Daniel Craig", birthdate=date(1968, 3, 2))
daniel_craig_id = daniel_craig.id
p2 = Person.objects.create(name="Anya Taylor-Joy", birthdate=date(1996, 4, 16))
p3 = Person.objects.create(name="Keanu Reeves", birthdate=date(1964, 9, 2))
p4 = Person.objects.create(name="Jean Gabin", birthdate=date(1904, 5, 17))
p5 = Person.objects.create(name="Chase Infiniti", birthdate=date(2000, 5, 5))
p6 = Person.objects.create(name="Arnold Schwarzenegger")

In [48]:
for p in daniel_craig, p2, p3, p4, p5, p6:
    print(p.name, p.birthdate, p.age())

Daniel Craig 1968-03-02 58
Anya Taylor-Joy 1996-04-16 30
Keanu Reeves 1964-09-02 61
Jean Gabin 1904-05-17 121
Chase Infiniti 2000-05-05 26
Arnold Schwarzenegger None None


### Films de Christopher Nolan

In [49]:
reset_queries()

# ── Réalisateur ──────────────────────────────────────────────────────────────
nolan = Person.objects.create(name="Christopher Nolan", birthdate=date(1970, 7, 30))
nolan_id = nolan.id

# ── Acteurs partagés entre plusieurs films ────────────────────────────────────
christian_bale = Person.objects.create(name="Christian Bale",       birthdate=date(1974,  1, 30))
michael_caine  = Person.objects.create(name="Michael Caine",        birthdate=date(1933,  3, 14))
gary_oldman    = Person.objects.create(name="Gary Oldman",          birthdate=date(1958,  3, 21))
tom_hardy      = Person.objects.create(name="Tom Hardy",            birthdate=date(1977,  9, 15))
joseph_gl      = Person.objects.create(name="Joseph Gordon-Levitt", birthdate=date(1981,  2, 17))
anne_hathaway  = Person.objects.create(name="Anne Hathaway",        birthdate=date(1982, 11, 12))
cillian_murphy = Person.objects.create(name="Cillian Murphy",       birthdate=date(1976,  5, 25))
kenneth_b      = Person.objects.create(name="Kenneth Branagh",      birthdate=date(1960, 12, 10))
matt_damon     = Person.objects.create(name="Matt Damon",           birthdate=date(1970, 10,  8))

# ── Memento (2000) ────────────────────────────────────────────────────────────
guy_pearce  = Person.objects.create(name="Guy Pearce",       birthdate=date(1967, 10,  5))
carrie_moss = Person.objects.create(name="Carrie-Anne Moss", birthdate=date(1967,  8, 21))
joe_panto   = Person.objects.create(name="Joe Pantoliano",   birthdate=date(1951,  9, 12))
memento = Movie.objects.create(title="Memento", year=2000, duration=113, director=nolan)
Play.objects.create(movie=memento, actor=guy_pearce,  character="Leonard Shelby")
Play.objects.create(movie=memento, actor=carrie_moss, character="Natalie")
Play.objects.create(movie=memento, actor=joe_panto,   character="Teddy")

# ── Batman Begins (2005) ──────────────────────────────────────────────────────
liam_neeson  = Person.objects.create(name="Liam Neeson",  birthdate=date(1952,  6,  7))
katie_holmes = Person.objects.create(name="Katie Holmes", birthdate=date(1978, 12, 18))
batman_begins = Movie.objects.create(title="Batman Begins", year=2005, duration=140, director=nolan)
Play.objects.create(movie=batman_begins, actor=christian_bale, character="Bruce Wayne / Batman")
Play.objects.create(movie=batman_begins, actor=michael_caine,  character="Alfred Pennyworth")
Play.objects.create(movie=batman_begins, actor=liam_neeson,    character="Ra's al Ghul")
Play.objects.create(movie=batman_begins, actor=gary_oldman,    character="Jim Gordon")
Play.objects.create(movie=batman_begins, actor=katie_holmes,   character="Rachel Dawes")

# ── The Prestige (2006) ───────────────────────────────────────────────────────
hugh_jackman = Person.objects.create(name="Hugh Jackman",       birthdate=date(1968, 10, 12))
scarlett_j   = Person.objects.create(name="Scarlett Johansson", birthdate=date(1984, 11, 22))
prestige = Movie.objects.create(title="The Prestige", year=2006, duration=130, director=nolan)
Play.objects.create(movie=prestige, actor=christian_bale, character="Alfred Borden")
Play.objects.create(movie=prestige, actor=hugh_jackman,   character="Robert Angier")
Play.objects.create(movie=prestige, actor=scarlett_j,     character="Olivia Wenscombe")
Play.objects.create(movie=prestige, actor=michael_caine,  character="Cutter")

# ── The Dark Knight (2008) ────────────────────────────────────────────────────
heath_ledger  = Person.objects.create(name="Heath Ledger",  birthdate=date(1979,  4,  4), is_alive=False)
aaron_eckhart = Person.objects.create(name="Aaron Eckhart", birthdate=date(1968,  3, 12))
dark_knight = Movie.objects.create(title="The Dark Knight", year=2008, duration=152, director=nolan)
Play.objects.create(movie=dark_knight, actor=christian_bale, character="Bruce Wayne / Batman")
Play.objects.create(movie=dark_knight, actor=heath_ledger,   character="The Joker")
Play.objects.create(movie=dark_knight, actor=aaron_eckhart,  character="Harvey Dent / Two-Face")
Play.objects.create(movie=dark_knight, actor=michael_caine,  character="Alfred Pennyworth")
Play.objects.create(movie=dark_knight, actor=gary_oldman,    character="Jim Gordon")

# ── Inception (2010) ──────────────────────────────────────────────────────────
leo_dicaprio = Person.objects.create(name="Leonardo DiCaprio", birthdate=date(1974, 11, 11))
elliot_page  = Person.objects.create(name="Elliot Page",       birthdate=date(1987,  2, 21))
ken_watanabe = Person.objects.create(name="Ken Watanabe",      birthdate=date(1959, 10, 21))
inception = Movie.objects.create(title="Inception", year=2010, duration=148, director=nolan)
Play.objects.create(movie=inception, actor=leo_dicaprio, character="Cobb")
Play.objects.create(movie=inception, actor=joseph_gl,    character="Arthur")
Play.objects.create(movie=inception, actor=elliot_page,  character="Ariadne")
Play.objects.create(movie=inception, actor=tom_hardy,    character="Eames")
Play.objects.create(movie=inception, actor=ken_watanabe, character="Saito")

# ── The Dark Knight Rises (2012) ──────────────────────────────────────────────
dk_rises = Movie.objects.create(title="The Dark Knight Rises", year=2012, duration=164, director=nolan)
Play.objects.create(movie=dk_rises, actor=christian_bale, character="Bruce Wayne / Batman")
Play.objects.create(movie=dk_rises, actor=tom_hardy,      character="Bane")
Play.objects.create(movie=dk_rises, actor=anne_hathaway,  character="Selina Kyle / Catwoman")
Play.objects.create(movie=dk_rises, actor=joseph_gl,      character="John Blake")
Play.objects.create(movie=dk_rises, actor=gary_oldman,    character="Jim Gordon")

# ── Interstellar (2014) ───────────────────────────────────────────────────────
matthew_mc = Person.objects.create(name="Matthew McConaughey", birthdate=date(1969, 11,  4))
jessica_c  = Person.objects.create(name="Jessica Chastain",    birthdate=date(1977,  3, 24))
interstellar = Movie.objects.create(title="Interstellar", year=2014, duration=169, director=nolan)
Play.objects.create(movie=interstellar, actor=matthew_mc,    character="Cooper")
Play.objects.create(movie=interstellar, actor=anne_hathaway, character="Dr. Brand")
Play.objects.create(movie=interstellar, actor=jessica_c,     character="Murph (adulte)")
Play.objects.create(movie=interstellar, actor=michael_caine, character="Prof. Brand")
Play.objects.create(movie=interstellar, actor=matt_damon,    character="Dr. Mann")

# ── Dunkirk (2017) ────────────────────────────────────────────────────────────
fionn_w      = Person.objects.create(name="Fionn Whitehead", birthdate=date(1997,  7, 28))
mark_rylance = Person.objects.create(name="Mark Rylance",    birthdate=date(1960,  1, 18))
dunkirk = Movie.objects.create(title="Dunkirk", year=2017, duration=106, director=nolan)
Play.objects.create(movie=dunkirk, actor=fionn_w,        character="Tommy")
Play.objects.create(movie=dunkirk, actor=mark_rylance,   character="Mr. Dawson")
Play.objects.create(movie=dunkirk, actor=tom_hardy,      character="Farrier")
Play.objects.create(movie=dunkirk, actor=cillian_murphy, character="Shivering Soldier")
Play.objects.create(movie=dunkirk, actor=kenneth_b,      character="Commander Bolton")

# ── Tenet (2020) ─────────────────────────────────────────────────────────────
jd_washington = Person.objects.create(name="John David Washington", birthdate=date(1984,  7, 28))
robert_p      = Person.objects.create(name="Robert Pattinson",      birthdate=date(1986,  5, 13))
elizabeth_d   = Person.objects.create(name="Elizabeth Debicki",     birthdate=date(1990,  8, 24))
dimple_k      = Person.objects.create(name="Dimple Kapadia",        birthdate=date(1957,  6,  8))
tenet = Movie.objects.create(title="Tenet", year=2020, duration=150, director=nolan)
Play.objects.create(movie=tenet, actor=jd_washington, character="The Protagonist")
Play.objects.create(movie=tenet, actor=robert_p,      character="Neil")
Play.objects.create(movie=tenet, actor=elizabeth_d,   character="Kat")
Play.objects.create(movie=tenet, actor=kenneth_b,     character="Andrei Sator")
Play.objects.create(movie=tenet, actor=dimple_k,      character="Priya Singh")

# ── Oppenheimer (2023) ────────────────────────────────────────────────────────
emily_blunt   = Person.objects.create(name="Emily Blunt",       birthdate=date(1983,  2, 23))
rdj           = Person.objects.create(name="Robert Downey Jr.", birthdate=date(1965,  4,  4))
florence_pugh = Person.objects.create(name="Florence Pugh",     birthdate=date(1996,  1,  3))
oppenheimer = Movie.objects.create(title="Oppenheimer", year=2023, duration=180, director=nolan)
Play.objects.create(movie=oppenheimer, actor=cillian_murphy, character="J. Robert Oppenheimer")
Play.objects.create(movie=oppenheimer, actor=emily_blunt,    character="Katherine Oppenheimer")
Play.objects.create(movie=oppenheimer, actor=matt_damon,     character="Leslie Groves")
Play.objects.create(movie=oppenheimer, actor=rdj,            character="Lewis Strauss")
Play.objects.create(movie=oppenheimer, actor=florence_pugh,  character="Jean Tatlock")

# Debug queries
for q in connection.queries:
    print(f"{q['time']}s -> SQL = {q['sql']}")

0.001s -> SQL = INSERT INTO "app_people_person" ("name", "birthdate", "is_alive") VALUES ('Christopher Nolan', '1970-07-30', 1) RETURNING "app_people_person"."id"
0.001s -> SQL = INSERT INTO "app_people_person" ("name", "birthdate", "is_alive") VALUES ('Christian Bale', '1974-01-30', 1) RETURNING "app_people_person"."id"
0.001s -> SQL = INSERT INTO "app_people_person" ("name", "birthdate", "is_alive") VALUES ('Michael Caine', '1933-03-14', 1) RETURNING "app_people_person"."id"
0.001s -> SQL = INSERT INTO "app_people_person" ("name", "birthdate", "is_alive") VALUES ('Gary Oldman', '1958-03-21', 1) RETURNING "app_people_person"."id"
0.001s -> SQL = INSERT INTO "app_people_person" ("name", "birthdate", "is_alive") VALUES ('Tom Hardy', '1977-09-15', 1) RETURNING "app_people_person"."id"
0.001s -> SQL = INSERT INTO "app_people_person" ("name", "birthdate", "is_alive") VALUES ('Joseph Gordon-Levitt', '1981-02-17', 1) RETURNING "app_people_person"."id"
0.001s -> SQL = INSERT INTO "app_people_

### James Bond 007

#### Avec Daniel Craig

In [50]:
# ── Réalisateurs ──────────────────────────────────────────────────────────────
martin_campbell = Person.objects.create(name="Martin Campbell",    birthdate=date(1943, 10, 24))
marc_forster    = Person.objects.create(name="Marc Forster",       birthdate=date(1969, 11, 30))
sam_mendes      = Person.objects.create(name="Sam Mendes",         birthdate=date(1965,  8,  1))
cary_fukunaga   = Person.objects.create(name="Cary Joji Fukunaga", birthdate=date(1977,  7, 10))

# Casino Royale (2006)
casino_royale = Movie.objects.create(title="Casino Royale", year=2006, duration=144, director=martin_campbell)
Play.objects.create(movie=casino_royale, actor=daniel_craig, character="James Bond")

# Quantum of Solace (2008)
quantum = Movie.objects.create(title="Quantum of Solace", year=2008, duration=106, director=marc_forster)
Play.objects.create(movie=quantum, actor=daniel_craig, character="James Bond")

# Skyfall (2012)
skyfall = Movie.objects.create(title="Skyfall", year=2012, duration=143, director=sam_mendes)
Play.objects.create(movie=skyfall, actor=daniel_craig, character="James Bond")

# Spectre (2015)
spectre = Movie.objects.create(title="Spectre", year=2015, duration=148, director=sam_mendes)
Play.objects.create(movie=spectre, actor=daniel_craig, character="James Bond")

# No Time to Die (2021)
no_time_to_die = Movie.objects.create(title="No Time to Die", year=2021, duration=163, director=cary_fukunaga)
Play.objects.create(movie=no_time_to_die, actor=daniel_craig, character="James Bond")

<Play: Play object (1129)>

#### Autres James Bond

In [51]:
# ── Acteurs Bond ──────────────────────────────────────────────────────────────
sean_connery   = Person.objects.create(name="Sean Connery",   birthdate=date(1930,  8, 25), is_alive=False)
george_lazenby = Person.objects.create(name="George Lazenby", birthdate=date(1939,  9,  5))
roger_moore    = Person.objects.create(name="Roger Moore",    birthdate=date(1927, 10, 14), is_alive=False)
timothy_dalton = Person.objects.create(name="Timothy Dalton", birthdate=date(1944,  3, 21))
pierce_brosnan = Person.objects.create(name="Pierce Brosnan", birthdate=date(1953,  5, 16))

# ── Personnages récurrents de la saga ─────────────────────────────────────────
bernard_lee      = Person.objects.create(name="Bernard Lee",      birthdate=date(1908,  1, 10), is_alive=False)  # M (Connery/Lazenby/Moore)
lois_maxwell     = Person.objects.create(name="Lois Maxwell",     birthdate=date(1927,  2, 14), is_alive=False)  # Moneypenny (Connery → Moore)
desmond_llewelyn = Person.objects.create(name="Desmond Llewelyn", birthdate=date(1914,  9, 12), is_alive=False)  # Q
robert_brown     = Person.objects.create(name="Robert Brown",     birthdate=date(1921,  3, 23), is_alive=False)  # M (Dalton)
caroline_bliss   = Person.objects.create(name="Caroline Bliss",   birthdate=date(1961,  7,  9))                  # Moneypenny (Dalton)
judi_dench       = Person.objects.create(name="Judi Dench",       birthdate=date(1934, 12,  9))                  # M (Brosnan/Craig)
samantha_bond    = Person.objects.create(name="Samantha Bond",    birthdate=date(1961, 11, 27))                  # Moneypenny (Brosnan)

# ── Réalisateurs ─────────────────────────────────────────────────────────────
# martin_campbell déjà créé (Casino Royale)
terence_young = Person.objects.create(name="Terence Young", birthdate=date(1915,  6, 20), is_alive=False)
peter_hunt    = Person.objects.create(name="Peter R. Hunt", birthdate=date(1925,  3, 11), is_alive=False)
lewis_gilbert = Person.objects.create(name="Lewis Gilbert", birthdate=date(1920,  3,  6), is_alive=False)
john_glen     = Person.objects.create(name="John Glen",     birthdate=date(1932,  5, 15))

# ── Dr. No (1962) — Sean Connery ─────────────────────────────────────────────
ursula_andress = Person.objects.create(name="Ursula Andress",  birthdate=date(1936,  3, 19))
joseph_wiseman = Person.objects.create(name="Joseph Wiseman",  birthdate=date(1918,  5, 15), is_alive=False)
dr_no = Movie.objects.create(title="Dr. No", year=1962, duration=110, director=terence_young)
Play.objects.create(movie=dr_no, actor=sean_connery,   character="James Bond")
Play.objects.create(movie=dr_no, actor=ursula_andress, character="Honey Ryder")
Play.objects.create(movie=dr_no, actor=joseph_wiseman, character="Dr. No")
Play.objects.create(movie=dr_no, actor=bernard_lee,    character="M")
Play.objects.create(movie=dr_no, actor=lois_maxwell,   character="Moneypenny")

# ── On Her Majesty's Secret Service (1969) — George Lazenby ──────────────────
diana_rigg    = Person.objects.create(name="Diana Rigg",    birthdate=date(1938,  7, 20), is_alive=False)
telly_savalas = Person.objects.create(name="Telly Savalas", birthdate=date(1922,  1, 21), is_alive=False)
ohmss = Movie.objects.create(title="On Her Majesty's Secret Service", year=1969, duration=142, director=peter_hunt)
Play.objects.create(movie=ohmss, actor=george_lazenby, character="James Bond")
Play.objects.create(movie=ohmss, actor=diana_rigg,     character="Tracy Di Vicenzo")
Play.objects.create(movie=ohmss, actor=telly_savalas,  character="Blofeld")
Play.objects.create(movie=ohmss, actor=bernard_lee,    character="M")
Play.objects.create(movie=ohmss, actor=lois_maxwell,   character="Moneypenny")

# ── The Spy Who Loved Me (1977) — Roger Moore ─────────────────────────────────
barbara_bach = Person.objects.create(name="Barbara Bach",  birthdate=date(1947,  8, 27))
curt_jurgens = Person.objects.create(name="Curt Jurgens",  birthdate=date(1915, 12, 13), is_alive=False)
spy_loved_me = Movie.objects.create(title="The Spy Who Loved Me", year=1977, duration=125, director=lewis_gilbert)
Play.objects.create(movie=spy_loved_me, actor=roger_moore,      character="James Bond")
Play.objects.create(movie=spy_loved_me, actor=barbara_bach,     character="Anya Amasova")
Play.objects.create(movie=spy_loved_me, actor=curt_jurgens,     character="Karl Stromberg")
Play.objects.create(movie=spy_loved_me, actor=bernard_lee,      character="M")
Play.objects.create(movie=spy_loved_me, actor=desmond_llewelyn, character="Q")

# ── The Living Daylights (1987) — Timothy Dalton ─────────────────────────────
maryam_dabo   = Person.objects.create(name="Maryam d'Abo",  birthdate=date(1960, 12, 27))
jeroen_krabbe = Person.objects.create(name="Jeroen Krabbé", birthdate=date(1944, 10,  5))
living_day = Movie.objects.create(title="The Living Daylights", year=1987, duration=130, director=john_glen)
Play.objects.create(movie=living_day, actor=timothy_dalton,  character="James Bond")
Play.objects.create(movie=living_day, actor=maryam_dabo,     character="Kara Milovy")
Play.objects.create(movie=living_day, actor=jeroen_krabbe,   character="Georgi Koskov")
Play.objects.create(movie=living_day, actor=robert_brown,    character="M")
Play.objects.create(movie=living_day, actor=desmond_llewelyn, character="Q")

# ── GoldenEye (1995) — Pierce Brosnan ────────────────────────────────────────
# martin_campbell déjà créé
izabella_sc = Person.objects.create(name="Izabella Scorupco", birthdate=date(1970,  6,  4))
sean_bean   = Person.objects.create(name="Sean Bean",         birthdate=date(1959,  4, 17))
goldeneye = Movie.objects.create(title="GoldenEye", year=1995, duration=130, director=martin_campbell)
Play.objects.create(movie=goldeneye, actor=pierce_brosnan,   character="James Bond")
Play.objects.create(movie=goldeneye, actor=izabella_sc,      character="Natalya Simonova")
Play.objects.create(movie=goldeneye, actor=sean_bean,        character="Alec Trevelyan / 006")
Play.objects.create(movie=goldeneye, actor=judi_dench,       character="M")
Play.objects.create(movie=goldeneye, actor=desmond_llewelyn, character="Q")

<Play: Play object (1154)>

#### Compléments acteurs 007 Daniel Craig

In [52]:
# ── Personnages récurrents — ère Daniel Craig ─────────────────────────────────
# judi_dench déjà créée (M dans l'ère Brosnan)
ralph_fiennes   = Person.objects.create(name="Ralph Fiennes",    birthdate=date(1962, 12, 22))  # M (Skyfall →)
ben_whishaw     = Person.objects.create(name="Ben Whishaw",      birthdate=date(1980, 10, 14))  # Q (Skyfall →)
naomie_harris   = Person.objects.create(name="Naomie Harris",    birthdate=date(1976,  9,  6))  # Moneypenny (Skyfall →)
jeffrey_wright  = Person.objects.create(name="Jeffrey Wright",   birthdate=date(1965, 12,  7))  # Felix Leiter
christoph_waltz = Person.objects.create(name="Christoph Waltz",  birthdate=date(1956, 10,  4))  # Blofeld

# Casino Royale (2006)
Play.objects.create(movie=casino_royale, actor=judi_dench,    character="M")
Play.objects.create(movie=casino_royale, actor=jeffrey_wright, character="Felix Leiter")

# Quantum of Solace (2008)
Play.objects.create(movie=quantum, actor=judi_dench,     character="M")
Play.objects.create(movie=quantum, actor=jeffrey_wright, character="Felix Leiter")

# Skyfall (2012) — transition M : Dench → Fiennes
Play.objects.create(movie=skyfall, actor=judi_dench,   character="M")
Play.objects.create(movie=skyfall, actor=ralph_fiennes, character="Gareth Mallory / M")
Play.objects.create(movie=skyfall, actor=ben_whishaw,   character="Q")
Play.objects.create(movie=skyfall, actor=naomie_harris, character="Eve Moneypenny")

# Spectre (2015)
Play.objects.create(movie=spectre, actor=ralph_fiennes,   character="M")
Play.objects.create(movie=spectre, actor=ben_whishaw,     character="Q")
Play.objects.create(movie=spectre, actor=naomie_harris,   character="Moneypenny")
Play.objects.create(movie=spectre, actor=christoph_waltz, character="Ernst Stavro Blofeld")

# No Time to Die (2021)
Play.objects.create(movie=no_time_to_die, actor=ralph_fiennes,   character="M")
Play.objects.create(movie=no_time_to_die, actor=ben_whishaw,     character="Q")
Play.objects.create(movie=no_time_to_die, actor=naomie_harris,   character="Moneypenny")
Play.objects.create(movie=no_time_to_die, actor=jeffrey_wright,  character="Felix Leiter")
Play.objects.create(movie=no_time_to_die, actor=christoph_waltz, character="Ernst Stavro Blofeld")

<Play: Play object (1171)>

### Autres films de Di Caprio

In [53]:
# ── Réalisateurs ──────────────────────────────────────────────────────────────
martin_scorsese  = Person.objects.create(name="Martin Scorsese",       birthdate=date(1942, 11, 17))
james_cameron    = Person.objects.create(name="James Cameron",         birthdate=date(1954,  8, 16))
steven_spielberg = Person.objects.create(name="Steven Spielberg",      birthdate=date(1946, 12, 18))
quentin_tt       = Person.objects.create(name="Quentin Tarantino",     birthdate=date(1963,  3, 27))
alejandro_gi     = Person.objects.create(name="Alejandro G. Iñárritu", birthdate=date(1963,  8, 15))

# ── The Aviator (2004) — Scorsese ─────────────────────────────────────────────
cate_blanchett  = Person.objects.create(name="Cate Blanchett",  birthdate=date(1969,  5, 14))
kate_beckinsale = Person.objects.create(name="Kate Beckinsale", birthdate=date(1973,  7, 26))
alec_baldwin    = Person.objects.create(name="Alec Baldwin",    birthdate=date(1958,  4,  3))
aviator = Movie.objects.create(title="The Aviator", year=2004, duration=170, director=martin_scorsese)
Play.objects.create(movie=aviator, actor=leo_dicaprio,   character="Howard Hughes")
Play.objects.create(movie=aviator, actor=cate_blanchett, character="Katharine Hepburn")
Play.objects.create(movie=aviator, actor=kate_beckinsale, character="Ava Gardner")
Play.objects.create(movie=aviator, actor=alec_baldwin,   character="Juan Trippe")

# ── The Wolf of Wall Street (2013) — Scorsese ─────────────────────────────────
jonah_hill    = Person.objects.create(name="Jonah Hill",    birthdate=date(1983, 12, 20))
margot_robbie = Person.objects.create(name="Margot Robbie", birthdate=date(1990,  7,  2))
wolf_wst = Movie.objects.create(title="The Wolf of Wall Street", year=2013, duration=180, director=martin_scorsese)
Play.objects.create(movie=wolf_wst, actor=leo_dicaprio, character="Jordan Belfort")
Play.objects.create(movie=wolf_wst, actor=jonah_hill,   character="Donnie Azoff")
Play.objects.create(movie=wolf_wst, actor=margot_robbie, character="Naomi Lapaglia")

# ── Titanic (1997) — James Cameron ────────────────────────────────────────────
kate_winslet = Person.objects.create(name="Kate Winslet", birthdate=date(1975, 10,  5))
billy_zane   = Person.objects.create(name="Billy Zane",   birthdate=date(1966,  2, 24))
titanic = Movie.objects.create(title="Titanic", year=1997, duration=194, director=james_cameron)
Play.objects.create(movie=titanic, actor=leo_dicaprio, character="Jack Dawson")
Play.objects.create(movie=titanic, actor=kate_winslet, character="Rose DeWitt Bukater")
Play.objects.create(movie=titanic, actor=billy_zane,   character="Cal Hockley")

# ── Catch Me If You Can (2002) — Spielberg ────────────────────────────────────
tom_hanks          = Person.objects.create(name="Tom Hanks",          birthdate=date(1956,  7,  9))
christopher_walken = Person.objects.create(name="Christopher Walken", birthdate=date(1943,  3, 31))
catch_me = Movie.objects.create(title="Catch Me If You Can", year=2002, duration=141, director=steven_spielberg)
Play.objects.create(movie=catch_me, actor=leo_dicaprio,      character="Frank Abagnale Jr.")
Play.objects.create(movie=catch_me, actor=tom_hanks,         character="Carl Hanratty")
Play.objects.create(movie=catch_me, actor=christopher_walken, character="Frank Abagnale Sr.")

# ── Django Unchained (2012) — Tarantino ───────────────────────────────────────
jamie_foxx      = Person.objects.create(name="Jamie Foxx",         birthdate=date(1967, 12, 13))
# christoph_waltz = Person.objects.create(name="Christoph Waltz",    birthdate=date(1956, 10,  4))
samuel_l_j      = Person.objects.create(name="Samuel L. Jackson",  birthdate=date(1948, 12, 21))
django_movie = Movie.objects.create(title="Django Unchained", year=2012, duration=165, director=quentin_tt)
Play.objects.create(movie=django_movie, actor=jamie_foxx,      character="Django Freeman")
Play.objects.create(movie=django_movie, actor=leo_dicaprio,    character="Calvin Candie")
Play.objects.create(movie=django_movie, actor=christoph_waltz, character="Dr. King Schultz")
Play.objects.create(movie=django_movie, actor=samuel_l_j,      character="Stephen")

# ── The Revenant (2015) — Iñárritu ───────────────────────────────────────────
domhnall_gleeson = Person.objects.create(name="Domhnall Gleeson", birthdate=date(1983,  5, 12))
will_poulter     = Person.objects.create(name="Will Poulter",     birthdate=date(1993,  1, 28))
revenant = Movie.objects.create(title="The Revenant", year=2015, duration=156, director=alejandro_gi)
Play.objects.create(movie=revenant, actor=leo_dicaprio,     character="Hugh Glass")
Play.objects.create(movie=revenant, actor=tom_hardy,        character="John Fitzgerald")  # déjà créé (Nolan)
Play.objects.create(movie=revenant, actor=domhnall_gleeson, character="Andrew Henry")
Play.objects.create(movie=revenant, actor=will_poulter,     character="Jim Bridger")


<Play: Play object (1192)>

### Autre film de Christian BAle

In [90]:
# ── American Psycho (2000) — Mary Harron ──────────────────────────────────────
mary_harron    = Person.objects.create(name="Mary Harron",    birthdate=date(1953,  1, 12))
jared_leto     = Person.objects.create(name="Jared Leto",     birthdate=date(1971, 12, 26))
willem_dafoe   = Person.objects.create(name="Willem Dafoe",   birthdate=date(1955,  7, 22))
reese_withers  = Person.objects.create(name="Reese Witherspoon", birthdate=date(1976,  3, 22))

american_psycho = Movie.objects.create(title="American Psycho", year=2000, duration=102, director=mary_harron)
Play.objects.create(movie=american_psycho, actor=christian_bale, character="Patrick Bateman")  # déjà créé (Nolan)
Play.objects.create(movie=american_psycho, actor=jared_leto,     character="Paul Allen")
Play.objects.create(movie=american_psycho, actor=willem_dafoe,   character="Donald Kimball")
Play.objects.create(movie=american_psycho, actor=reese_withers,  character="Evelyn Williams")


<Play: Play object (1239)>

### Films à couteaux tirés
DANIEL CRAIG — Détective Benoit Blanc (saga Knives Out)

In [54]:
rian_johnson = Person.objects.create(name="Rian Johnson", birthdate=date(1973, 12, 17))

# ── Knives Out (2019) ─────────────────────────────────────────────────────────
ana_de_armas      = Person.objects.create(name="Ana de Armas",      birthdate=date(1988,  4, 30))
chris_evans       = Person.objects.create(name="Chris Evans",       birthdate=date(1981,  6, 13))
jamie_lee_curtis  = Person.objects.create(name="Jamie Lee Curtis",  birthdate=date(1958, 11, 22))
christopher_plum  = Person.objects.create(name="Christopher Plummer", birthdate=date(1929, 12, 13), is_alive=False)
knives_out = Movie.objects.create(title="Knives Out", year=2019, duration=131, director=rian_johnson)
Play.objects.create(movie=knives_out, actor=daniel_craig,    character="Benoit Blanc")
Play.objects.create(movie=knives_out, actor=ana_de_armas,    character="Marta Cabrera")
Play.objects.create(movie=knives_out, actor=chris_evans,     character="Ransom Drysdale")
Play.objects.create(movie=knives_out, actor=jamie_lee_curtis, character="Linda Drysdale")
Play.objects.create(movie=knives_out, actor=christopher_plum, character="Harlan Thrombey")

# ── Glass Onion : Une histoire à couteaux tirés (2022) — Netflix ──────────────
edward_norton = Person.objects.create(name="Edward Norton",  birthdate=date(1969,  8, 18))
janelle_monae = Person.objects.create(name="Janelle Monáe",  birthdate=date(1985, 12,  1))
kate_hudson   = Person.objects.create(name="Kate Hudson",    birthdate=date(1979,  4, 19))
dave_bautista = Person.objects.create(name="Dave Bautista",  birthdate=date(1969,  1, 18))
glass_onion = Movie.objects.create(title="Glass Onion", year=2022, duration=139, director=rian_johnson)
Play.objects.create(movie=glass_onion, actor=daniel_craig,  character="Benoit Blanc")
Play.objects.create(movie=glass_onion, actor=edward_norton, character="Miles Bron")
Play.objects.create(movie=glass_onion, actor=janelle_monae, character="Andi Brand / Helen Brand")
Play.objects.create(movie=glass_onion, actor=kate_hudson,   character="Birdie Jay")
Play.objects.create(movie=glass_onion, actor=dave_bautista, character="Duke Cody")

# ── Wake Up Dead Man (2025) — Netflix ─────────────────────────────────────────
glenn_close   = Person.objects.create(name="Glenn Close",   birthdate=date(1947,  3, 19))
josh_brolin   = Person.objects.create(name="Josh Brolin",   birthdate=date(1968,  2, 12))
mila_kunis    = Person.objects.create(name="Mila Kunis",    birthdate=date(1983,  8, 14))
jeremy_renner = Person.objects.create(name="Jeremy Renner", birthdate=date(1971,  1,  7))
josh_oconnor  = Person.objects.create(name="Josh O'Connor", birthdate=date(1990,  5, 20))
wake_up = Movie.objects.create(title="Wake Up Dead Man", year=2025, duration=144, director=rian_johnson)
Play.objects.create(movie=wake_up, actor=daniel_craig,  character="Benoit Blanc")
Play.objects.create(movie=wake_up, actor=glenn_close,   character="Martha Delacroix")
Play.objects.create(movie=wake_up, actor=josh_brolin,   character="M. Jefferson Wicks")
Play.objects.create(movie=wake_up, actor=mila_kunis,    character="Chief Geraldine Scott")
Play.objects.create(movie=wake_up, actor=jeremy_renner, character="Dr. Nat Sharp")
Play.objects.create(movie=wake_up, actor=josh_oconnor,  character="Fr. Jud Duplenticy")

<Play: Play object (1208)>

### Clint EASTWOOD
acteur et/ou réalisateur

In [55]:
clint_eastwood = Person.objects.create(name="Clint Eastwood", birthdate=date(1930,  5, 31))
clint_eastwood_id = clint_eastwood.id
sergio_leone   = Person.objects.create(name="Sergio Leone",   birthdate=date(1929,  1,  3), is_alive=False)
don_siegel     = Person.objects.create(name="Don Siegel",     birthdate=date(1912, 10, 26), is_alive=False)

# ── Le Bon, la Brute et le Truand (1966) — Sergio Leone ──────────────────────
lee_van_cleef = Person.objects.create(name="Lee Van Cleef", birthdate=date(1925,  1,  9), is_alive=False)
eli_wallach   = Person.objects.create(name="Eli Wallach",   birthdate=date(1915, 12,  7), is_alive=False)
gbu = Movie.objects.create(title="Le Bon, la Brute et le Truand", year=1966, duration=161, director=sergio_leone)
Play.objects.create(movie=gbu, actor=clint_eastwood, character="Blondie / Le Bon")
Play.objects.create(movie=gbu, actor=lee_van_cleef,  character="Angel Eyes / Le Truand")
Play.objects.create(movie=gbu, actor=eli_wallach,    character="Tuco / La Brute")

# ── L'Inspecteur Harry (1971) — Don Siegel ───────────────────────────────────
andy_robinson = Person.objects.create(name="Andy Robinson", birthdate=date(1942,  2, 14))
dirty_harry = Movie.objects.create(title="L'Inspecteur Harry", year=1971, duration=102, director=don_siegel)
Play.objects.create(movie=dirty_harry, actor=clint_eastwood, character="Harry Callahan")
Play.objects.create(movie=dirty_harry, actor=andy_robinson,  character="Scorpio")

# ── Impitoyable (1992) — Clint Eastwood réalisateur et acteur ────────────────
gene_hackman   = Person.objects.create(name="Gene Hackman",   birthdate=date(1930,  1, 30), is_alive=False)
morgan_freeman = Person.objects.create(name="Morgan Freeman", birthdate=date(1937,  6,  1))
richard_harris = Person.objects.create(name="Richard Harris", birthdate=date(1930, 10,  1), is_alive=False)
unforgiven = Movie.objects.create(title="Impitoyable", year=1992, duration=130, director=clint_eastwood)
Play.objects.create(movie=unforgiven, actor=clint_eastwood, character="William Munny")
Play.objects.create(movie=unforgiven, actor=gene_hackman,   character="Little Bill Daggett")
Play.objects.create(movie=unforgiven, actor=morgan_freeman, character="Ned Logan")
Play.objects.create(movie=unforgiven, actor=richard_harris, character="English Bob")

# ── Gran Torino (2008) — Clint Eastwood réalisateur et acteur ────────────────
bee_vang  = Person.objects.create(name="Bee Vang",  birthdate=date(1992,  5, 25))
ahney_her = Person.objects.create(name="Ahney Her", birthdate=date(1991, 11, 16))
gran_torino = Movie.objects.create(title="Gran Torino", year=2008, duration=116, director=clint_eastwood)
Play.objects.create(movie=gran_torino, actor=clint_eastwood, character="Walt Kowalski")
Play.objects.create(movie=gran_torino, actor=bee_vang,       character="Thao")
Play.objects.create(movie=gran_torino, actor=ahney_her,      character="Sue")

# ── Juré n°2 (2024) — Clint Eastwood réalisateur (non acteur) ────────────────
nicholas_hoult = Person.objects.create(name="Nicholas Hoult", birthdate=date(1989, 12,  7))
toni_collette  = Person.objects.create(name="Toni Collette",  birthdate=date(1972, 11,  1))
jk_simmons     = Person.objects.create(name="J.K. Simmons",   birthdate=date(1955,  1,  9))
jure2 = Movie.objects.create(title="Juré n°2", year=2024, duration=114, director=clint_eastwood)
Play.objects.create(movie=jure2, actor=nicholas_hoult, character="Justin Kemp")
Play.objects.create(movie=jure2, actor=toni_collette,  character="Faith Killebrew")
Play.objects.create(movie=jure2, actor=jk_simmons,     character="Harold")

<Play: Play object (1223)>

### Steve McQUEEN — homonymes
- acteur (1930-1980)
- réalisateur (né 1969)

In [56]:
steve_mcqueen_actor = Person.objects.create(name="Steve McQueen", birthdate=date(1930,  3, 24), is_alive=False)
steve_mcqueen_dir   = Person.objects.create(name="Steve McQueen", birthdate=date(1969, 10,  9))
# ⚠ Person.objects.get(name="Steve McQueen") => MultipleObjectsReturned !
# => distinguer par birthdate ou par id : steve_mcqueen_actor.id / steve_mcqueen_dir.id

john_sturges = Person.objects.create(name="John Sturges", birthdate=date(1910,  1,  3), is_alive=False)
peter_yates  = Person.objects.create(name="Peter Yates",  birthdate=date(1929,  7, 24), is_alive=False)

# ── La Grande Évasion (1963) — Steve McQueen acteur ──────────────────────────
james_garner      = Person.objects.create(name="James Garner",         birthdate=date(1928,  4,  7), is_alive=False)
richard_attenboro = Person.objects.create(name="Richard Attenborough", birthdate=date(1923,  8, 29), is_alive=False)
great_escape = Movie.objects.create(title="La Grande Évasion", year=1963, duration=172, director=john_sturges)
Play.objects.create(movie=great_escape, actor=steve_mcqueen_actor, character="Virgil Hilts")
Play.objects.create(movie=great_escape, actor=james_garner,        character="Bob Hendley")
Play.objects.create(movie=great_escape, actor=richard_attenboro,   character="Roger Bartlett")

# ── Bullitt (1968) — Steve McQueen acteur ────────────────────────────────────
robert_vaughn = Person.objects.create(name="Robert Vaughn",    birthdate=date(1932, 11, 22), is_alive=False)
jacqueline_b  = Person.objects.create(name="Jacqueline Bisset", birthdate=date(1944,  9, 13))
bullitt = Movie.objects.create(title="Bullitt", year=1968, duration=114, director=peter_yates)
Play.objects.create(movie=bullitt, actor=steve_mcqueen_actor, character="Frank Bullitt")
Play.objects.create(movie=bullitt, actor=robert_vaughn,       character="Walter Chalmers")
Play.objects.create(movie=bullitt, actor=jacqueline_b,        character="Cathy")

# ── Shame (2011) — Steve McQueen réalisateur ─────────────────────────────────
michael_fassbender = Person.objects.create(name="Michael Fassbender", birthdate=date(1977,  4,  2))
carey_mulligan     = Person.objects.create(name="Carey Mulligan",     birthdate=date(1985,  5, 28))
shame = Movie.objects.create(title="Shame", year=2011, duration=101, director=steve_mcqueen_dir)
Play.objects.create(movie=shame, actor=michael_fassbender, character="Brandon Sullivan")
Play.objects.create(movie=shame, actor=carey_mulligan,     character="Sissy Sullivan")

# ── Widows (2018) — Steve McQueen réalisateur ────────────────────────────────
viola_davis   = Person.objects.create(name="Viola Davis",        birthdate=date(1965,  8, 11))
michelle_rod  = Person.objects.create(name="Michelle Rodriguez", birthdate=date(1978,  7, 12))
colin_farrell = Person.objects.create(name="Colin Farrell",      birthdate=date(1976,  5, 31))
# elizabeth_d déjà créée (section Nolan — Tenet)
widows = Movie.objects.create(title="Widows", year=2018, duration=129, director=steve_mcqueen_dir)
Play.objects.create(movie=widows, actor=viola_davis,   character="Veronica Rawlings")
Play.objects.create(movie=widows, actor=michelle_rod,  character="Linda")
Play.objects.create(movie=widows, actor=elizabeth_d,   character="Alice")
Play.objects.create(movie=widows, actor=colin_farrell, character="Jack Mulligan")

<Play: Play object (1235)>

## Queries

### QuerySet et show SQL

In [57]:
Person.objects.all()

<QuerySet [<Person: Person object (917)>, <Person: Person object (918)>, <Person: Person object (919)>, <Person: Person object (920)>, <Person: Person object (921)>, <Person: Person object (922)>, <Person: Person object (923)>, <Person: Person object (924)>, <Person: Person object (925)>, <Person: Person object (926)>, <Person: Person object (927)>, <Person: Person object (928)>, <Person: Person object (929)>, <Person: Person object (930)>, <Person: Person object (931)>, <Person: Person object (932)>, <Person: Person object (933)>, <Person: Person object (934)>, <Person: Person object (935)>, <Person: Person object (936)>, '...(remaining elements truncated)...']>

In [58]:
reset_queries()
qs = Person.objects.filter(name="Daniel Craig")
connection.queries

[]

In [59]:
type(qs) # Lazy object

django.db.models.query.QuerySet

In [60]:
list(qs) # execute query + iterate result

[<Person: Person object (917)>]

In [61]:
print(qs)
connection.queries

<QuerySet [<Person: Person object (917)>]>


[{'sql': 'SELECT "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_people_person" WHERE "app_people_person"."name" = \'Daniel Craig\'',
  'time': '0.000'}]

In [62]:
reset_queries()
print(Person.objects.filter(name="Daniel Craig")) # => call str()
connection.queries

<QuerySet [<Person: Person object (917)>]>


[{'sql': 'SELECT "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_people_person" WHERE "app_people_person"."name" = \'Daniel Craig\' LIMIT 21',
  'time': '0.000'}]

In [63]:
reset_queries()
qs = Person.objects.filter(name="Daniel Craig")
connection.queries

[]

In [64]:
it = iter(qs) # execute query
connection.queries

[{'sql': 'SELECT "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_people_person" WHERE "app_people_person"."name" = \'Daniel Craig\'',
  'time': '0.000'}]

In [65]:
next(it) # next row from cursor => new object Person

<Person: Person object (917)>

In [66]:
try:
    next(it) # fin iteration
except StopIteration:
    print("This is the end")

This is the end


In [67]:
reset_queries()
qs = Person.objects.filter(name="Daniel Craig")
print('Before iteration:', connection.queries)
for i, p in enumerate(qs, start=1):
    print('Next iteration:', connection.queries)
    print(i, p)
print('End iteration:', connection.queries)

Before iteration: []
Next iteration: [{'sql': 'SELECT "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_people_person" WHERE "app_people_person"."name" = \'Daniel Craig\'', 'time': '0.000'}]
1 Person object (917)
End iteration: [{'sql': 'SELECT "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_people_person" WHERE "app_people_person"."name" = \'Daniel Craig\'', 'time': '0.000'}]


In [68]:
reset_queries()
qs = Person.objects.all()
print('Before iteration:', connection.queries)
for i, p in enumerate(qs, start=1):
    print('Next iteration:', connection.queries)
    print(i, p)
print('End iteration:', connection.queries)

Before iteration: []
Next iteration: [{'sql': 'SELECT "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_people_person"', 'time': '0.001'}]
1 Person object (917)
Next iteration: [{'sql': 'SELECT "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_people_person"', 'time': '0.001'}]
2 Person object (918)
Next iteration: [{'sql': 'SELECT "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_people_person"', 'time': '0.001'}]
3 Person object (919)
Next iteration: [{'sql': 'SELECT "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_people_person"', 'time': '0.001'}]
4 Person object (920)
Next iteration: [{'sql': 'SELECT "app_people_person"."id", "app_people_person"."name", "app_people_person".

In [69]:
reset_queries()
print('movie (in memory):', inception.title, inception.year)
print(connection.queries)
director = inception.director
print('director (in memory):', director.name)
print(connection.queries)

movie (in memory): Inception 2010
[]
director (in memory): Christopher Nolan
[]


### N + 1 Queries (non optimisé)

In [70]:
movie_id = inception.id
movie_id

275

In [71]:
# reset cache ORM
connection.close()
reset_queries()
movie = Movie.objects.get(pk=movie_id)
print(f'* SQL[{len(connection.queries)}]: {connection.queries}')
print('* movie read from db:', movie, movie.title)
director = movie.director
print(f'* SQL[{len(connection.queries)}]: {connection.queries}')
print('* director:', director, director.name)

# Bilan: SQL => 1 + 1

* SQL[1]: [{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year", "app_movie_movie"."duration", "app_movie_movie"."synopsis", "app_movie_movie"."director_id" FROM "app_movie_movie" WHERE "app_movie_movie"."id" = 275 LIMIT 21', 'time': '0.000'}]
* movie read from db: Movie object (275) Inception
* SQL[2]: [{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year", "app_movie_movie"."duration", "app_movie_movie"."synopsis", "app_movie_movie"."director_id" FROM "app_movie_movie" WHERE "app_movie_movie"."id" = 275 LIMIT 21', 'time': '0.000'}, {'sql': 'SELECT "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_people_person" WHERE "app_people_person"."id" = 923 LIMIT 21', 'time': '0.000'}]
* director: Person object (923) Christopher Nolan


In [72]:
# reset cache ORM
connection.close()
reset_queries()
qs = Movie.objects.filter(year__gt=2000)
print(f'* SQL[{len(connection.queries)}]: {connection.queries}')
print()
for movie in qs:
    print(f'* SQL[{len(connection.queries)}]: {connection.queries}')
    print('* movie read from db:', movie, movie.title, movie.director_id)
    director = movie.director # SQL : + 1 query
    print(f'* SQL[{len(connection.queries)}]: {connection.queries}')
    print('* director:', director, director.name)
    print()

# Bilan: SQL => 1 + 1

* SQL[0]: []

* SQL[1]: [{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year", "app_movie_movie"."duration", "app_movie_movie"."synopsis", "app_movie_movie"."director_id" FROM "app_movie_movie" WHERE "app_movie_movie"."year" > 2000', 'time': '0.000'}]
* movie read from db: Movie object (272) Batman Begins 923
* SQL[2]: [{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year", "app_movie_movie"."duration", "app_movie_movie"."synopsis", "app_movie_movie"."director_id" FROM "app_movie_movie" WHERE "app_movie_movie"."year" > 2000', 'time': '0.000'}, {'sql': 'SELECT "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_people_person" WHERE "app_people_person"."id" = 923 LIMIT 21', 'time': '0.000'}]
* director: Person object (923) Christopher Nolan

* SQL[2]: [{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."

### N+ 1 (optimisé)

In [73]:
connection.close()
reset_queries()
movie = Movie.objects.prefetch_related('director').get(pk=movie_id)
print(f'* SQL[{len(connection.queries)}]: {connection.queries}')
print('* movie read from db:', movie, movie.title)
director = movie.director
print(f'* SQL[{len(connection.queries)}]: {connection.queries}')
print('* director:', director, director.name)

* SQL[2]: [{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year", "app_movie_movie"."duration", "app_movie_movie"."synopsis", "app_movie_movie"."director_id" FROM "app_movie_movie" WHERE "app_movie_movie"."id" = 275 LIMIT 21', 'time': '0.000'}, {'sql': 'SELECT "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_people_person" WHERE "app_people_person"."id" = 923', 'time': '0.000'}]
* movie read from db: Movie object (275) Inception
* SQL[2]: [{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year", "app_movie_movie"."duration", "app_movie_movie"."synopsis", "app_movie_movie"."director_id" FROM "app_movie_movie" WHERE "app_movie_movie"."id" = 275 LIMIT 21', 'time': '0.000'}, {'sql': 'SELECT "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_people_person" WHERE "a

In [74]:
connection.close()
reset_queries()
qs = Movie.objects.select_related('director').filter(year__gt=2000) # 1 query JOIN
print(f'* SQL[{len(connection.queries)}]: {connection.queries}')
print()
for movie in qs:
    print(f'* SQL[{len(connection.queries)}]: {connection.queries}')
    print('* movie read from db:', movie, movie.title, movie.director_id)
    director = movie.director # SQL : + 1 query
    print(f'* SQL[{len(connection.queries)}]: {connection.queries}')
    print('* director:', director, director.name)
    print()

* SQL[0]: []

* SQL[1]: [{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year", "app_movie_movie"."duration", "app_movie_movie"."synopsis", "app_movie_movie"."director_id", "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_movie_movie" LEFT OUTER JOIN "app_people_person" ON ("app_movie_movie"."director_id" = "app_people_person"."id") WHERE "app_movie_movie"."year" > 2000', 'time': '0.000'}]
* movie read from db: Movie object (272) Batman Begins 923
* SQL[1]: [{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year", "app_movie_movie"."duration", "app_movie_movie"."synopsis", "app_movie_movie"."director_id", "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_movie_movie" LEFT OUTER JOIN "app_people_person" ON ("app_movie_movie"."director_id" = "app_people_person"

In [75]:
connection.close()
reset_queries()
qs = Movie.objects.prefetch_related('director').filter(year__gt=2000) # 2 queries
print(f'* SQL[{len(connection.queries)}]: {connection.queries}')
print()
for movie in qs:
    print(f'* SQL[{len(connection.queries)}]: {connection.queries}')
    print('* movie read from db:', movie, movie.title, movie.director_id)
    director = movie.director # SQL : + 1 query
    print(f'* SQL[{len(connection.queries)}]: {connection.queries}')
    print('* director:', director, director.name)
    print()

* SQL[0]: []

* SQL[2]: [{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year", "app_movie_movie"."duration", "app_movie_movie"."synopsis", "app_movie_movie"."director_id" FROM "app_movie_movie" WHERE "app_movie_movie"."year" > 2000', 'time': '0.000'}, {'sql': 'SELECT "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_people_person" WHERE ("app_people_person"."id" = 923 OR "app_people_person"."id" = 956 OR "app_people_person"."id" = 957 OR "app_people_person"."id" = 958 OR "app_people_person"."id" = 959 OR "app_people_person"."id" = 991 OR "app_people_person"."id" = 993 OR "app_people_person"."id" = 994 OR "app_people_person"."id" = 995 OR "app_people_person"."id" = 1009 OR "app_people_person"."id" = 1023 OR "app_people_person"."id" = 1038)', 'time': '0.000'}]
* movie read from db: Movie object (272) Batman Begins 923
* SQL[2]: [{'sql': 'SELECT "app_movie_movie"."id", "app

In [76]:
connection.close()
reset_queries()
movie = Movie.objects \
    .select_related('director') \
    .prefetch_related('actors') \
    .get(pk=movie_id)
print(f'* SQL[{len(connection.queries)}]: {connection.queries}')
print('* movie read from db:', movie, movie.title)
director = movie.director
print(f'* SQL[{len(connection.queries)}]: {connection.queries}')
print('* director:', director, director.name)
for actor in movie.actors.all():
    print('* actor:', actor, actor.name)
print(f'* SQL[{len(connection.queries)}]: {connection.queries}')

* SQL[2]: [{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year", "app_movie_movie"."duration", "app_movie_movie"."synopsis", "app_movie_movie"."director_id", "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_movie_movie" LEFT OUTER JOIN "app_people_person" ON ("app_movie_movie"."director_id" = "app_people_person"."id") WHERE "app_movie_movie"."id" = 275 LIMIT 21', 'time': '0.000'}, {'sql': 'SELECT ("app_movie_play"."movie_id") AS "_prefetch_related_val_movie_id", "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive" FROM "app_people_person" INNER JOIN "app_movie_play" ON ("app_people_person"."id" = "app_movie_play"."actor_id") WHERE "app_movie_play"."movie_id" IN (275)', 'time': '0.000'}]
* movie read from db: Movie object (275) Inception
* SQL[2]: [{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."

In [77]:
connection.close()
reset_queries()
qs = Movie.objects\
    .select_related('director') \
    .prefetch_related('actors') \
    .filter(year__gt=2000) 
print(f'* SQL[{len(connection.queries)}]: {connection.queries}\n')
for movie in qs:
    print('* movie:', movie, movie.title, movie.director_id)
    director = movie.director # SQL : + 1 query
    print('* director:', director, director.name)
    for actor in movie.actors.all():
        print('* actor:', actor, actor.name)
    print()
print(f'* SQL[{len(connection.queries)}]: {connection.queries}\n')

* SQL[0]: []

* movie: Movie object (272) Batman Begins 923
* director: Person object (923) Christopher Nolan
* actor: Person object (924) Christian Bale
* actor: Person object (925) Michael Caine
* actor: Person object (936) Liam Neeson
* actor: Person object (926) Gary Oldman
* actor: Person object (937) Katie Holmes

* movie: Movie object (273) The Prestige 923
* director: Person object (923) Christopher Nolan
* actor: Person object (924) Christian Bale
* actor: Person object (938) Hugh Jackman
* actor: Person object (939) Scarlett Johansson
* actor: Person object (925) Michael Caine

* movie: Movie object (274) The Dark Knight 923
* director: Person object (923) Christopher Nolan
* actor: Person object (924) Christian Bale
* actor: Person object (940) Heath Ledger
* actor: Person object (941) Aaron Eckhart
* actor: Person object (925) Michael Caine
* actor: Person object (926) Gary Oldman

* movie: Movie object (275) Inception 923
* director: Person object (923) Christopher Nolan
*

### Navigation asso Play

#### Movie -> Actor

In [78]:
movie.actors.all()

<QuerySet [<Person: Person object (1047)>, <Person: Person object (1048)>, <Person: Person object (951)>, <Person: Person object (1049)>]>

In [79]:
movie.plays.all()

<QuerySet [<Play: Play object (1232)>, <Play: Play object (1233)>, <Play: Play object (1234)>, <Play: Play object (1235)>]>

In [80]:
for play in movie.plays.all():
    print(f'{play.actor.name}: {play.character}')

Viola Davis: Veronica Rawlings
Michelle Rodriguez: Linda
Elizabeth Debicki: Alice
Colin Farrell: Jack Mulligan


In [81]:
connection.close()
reset_queries()
qs = Movie.objects\
    .select_related('director') \
    .prefetch_related('plays__actor') \
    .filter(year__gt=2000) 
print(f'* SQL[{len(connection.queries)}]: {connection.queries}\n')
for movie in qs:
    print('* movie:', movie, movie.title, movie.director_id)
    director = movie.director # SQL : + 1 query
    print('* director:', director, director.name)
    for play in movie.plays.all():
        print(f'* actor: {play.actor.name} ({play.character})')
    print()
print(f'* SQL[{len(connection.queries)}]: {connection.queries}\n')

* SQL[0]: []

* movie: Movie object (272) Batman Begins 923
* director: Person object (923) Christopher Nolan
* actor: Christian Bale (Bruce Wayne / Batman)
* actor: Michael Caine (Alfred Pennyworth)
* actor: Liam Neeson (Ra's al Ghul)
* actor: Gary Oldman (Jim Gordon)
* actor: Katie Holmes (Rachel Dawes)

* movie: Movie object (273) The Prestige 923
* director: Person object (923) Christopher Nolan
* actor: Christian Bale (Alfred Borden)
* actor: Hugh Jackman (Robert Angier)
* actor: Scarlett Johansson (Olivia Wenscombe)
* actor: Michael Caine (Cutter)

* movie: Movie object (274) The Dark Knight 923
* director: Person object (923) Christopher Nolan
* actor: Christian Bale (Bruce Wayne / Batman)
* actor: Heath Ledger (The Joker)
* actor: Aaron Eckhart (Harvey Dent / Two-Face)
* actor: Michael Caine (Alfred Pennyworth)
* actor: Gary Oldman (Jim Gordon)

* movie: Movie object (275) Inception 923
* director: Person object (923) Christopher Nolan
* actor: Leonardo DiCaprio (Cobb)
* actor:

#### Person -> Movie

In [82]:
p = Person.objects.get(pk=nolan_id)
print(p.directed_movies.all())
print(p.played_movies.all())
print(p.plays.all())

<QuerySet [<Movie: Movie object (271)>, <Movie: Movie object (272)>, <Movie: Movie object (273)>, <Movie: Movie object (274)>, <Movie: Movie object (275)>, <Movie: Movie object (276)>, <Movie: Movie object (277)>, <Movie: Movie object (278)>, <Movie: Movie object (279)>, <Movie: Movie object (280)>]>
<QuerySet []>
<QuerySet []>


In [83]:
p = Person.objects.get(pk=daniel_craig_id)
print(p.directed_movies.all())
print(p.played_movies.all())
print(p.plays.all())

<QuerySet []>
<QuerySet [<Movie: Movie object (281)>, <Movie: Movie object (282)>, <Movie: Movie object (283)>, <Movie: Movie object (284)>, <Movie: Movie object (285)>, <Movie: Movie object (297)>, <Movie: Movie object (298)>, <Movie: Movie object (299)>]>
<QuerySet [<Play: Play object (1125)>, <Play: Play object (1126)>, <Play: Play object (1127)>, <Play: Play object (1128)>, <Play: Play object (1129)>, <Play: Play object (1193)>, <Play: Play object (1198)>, <Play: Play object (1203)>]>


In [84]:
p = Person.objects.get(pk=clint_eastwood_id)
print("Directed")
for m in p.directed_movies.all():
    print(f" * {m.title} ({m.year})")
print("Played")
for m in p.played_movies.all():
    print(f" * {m.title} ({m.year})")
print("Played (detail)")
for pl in p.plays.all():
    print(f" * {pl.movie.title} ({pl.movie.year}): {pl.character}")

Directed
 * Impitoyable (1992)
 * Gran Torino (2008)
 * Juré n°2 (2024)
Played
 * Le Bon, la Brute et le Truand (1966)
 * L'Inspecteur Harry (1971)
 * Impitoyable (1992)
 * Gran Torino (2008)
Played (detail)
 * Le Bon, la Brute et le Truand (1966): Blondie / Le Bon
 * L'Inspecteur Harry (1971): Harry Callahan
 * Impitoyable (1992): William Munny
 * Gran Torino (2008): Walt Kowalski


## Queries (advanced)

In [85]:
# TODO: les films avec daniel Craig avec leur realisateur (et ses co-acteurs)

In [86]:
# les films de realisateurs nés en 1930
# Les films dont les acteurs sont nés en 1930
# Les films où ont joué ensemble Di Caprio et Kate Winslet
# Les films de Nolan avec les acteurs 
#       et pour chacun le nombre de réalisateurs 
#       avec qui ils ont collaborés (hors Nolan)

In [96]:
qs = Movie.objects.filter(director__name='Christopher Nolan') \
    .select_related('director') \
    .prefetch_related('actors')

for o_movie in qs:
    print(f'* {o_movie.title} ({o_movie.year})')
    for o_actor in o_movie.actors.all():
        nb_real = Movie.objects.filter(actors__id=o_actor.id) \
            .order_by('director') \
            .exclude(director=o_movie.director) \
            .values('director') \
            .distinct() \
            .count()
        print(f'    - {o_actor.name} ({nb_real})')

* Memento (2000)
    - Guy Pearce (0)
    - Carrie-Anne Moss (0)
    - Joe Pantoliano (0)
* Batman Begins (2005)
    - Christian Bale (1)
    - Michael Caine (0)
    - Liam Neeson (0)
    - Gary Oldman (0)
    - Katie Holmes (0)
* The Prestige (2006)
    - Christian Bale (1)
    - Hugh Jackman (0)
    - Scarlett Johansson (0)
    - Michael Caine (0)
* The Dark Knight (2008)
    - Christian Bale (1)
    - Heath Ledger (0)
    - Aaron Eckhart (0)
    - Michael Caine (0)
    - Gary Oldman (0)
* Inception (2010)
    - Leonardo DiCaprio (5)
    - Joseph Gordon-Levitt (0)
    - Elliot Page (0)
    - Tom Hardy (1)
    - Ken Watanabe (0)
* The Dark Knight Rises (2012)
    - Christian Bale (1)
    - Tom Hardy (1)
    - Anne Hathaway (0)
    - Joseph Gordon-Levitt (0)
    - Gary Oldman (0)
* Interstellar (2014)
    - Matthew McConaughey (0)
    - Anne Hathaway (0)
    - Jessica Chastain (0)
    - Michael Caine (0)
    - Matt Damon (0)
* Dunkirk (2017)
    - Fionn Whitehead (0)
    - Mark Rylance

In [99]:
qs = Movie.objects.filter(director__name='Christopher Nolan') \
    .select_related('director') \
    .prefetch_related('actors')

for o_movie in qs:
    print(f'* {o_movie.title} ({o_movie.year})')
    # BUG: fix query de group by director not actor
    qs_other = o_movie.actors.all() \
        .annotate(nb_real=Count('directed_movies__director', distinct=True)) 
    for o_actor in qs_other:
        print(f'    - {o_actor.name} ({o_actor.nb_real})')

* Memento (2000)
    - Guy Pearce (0)
    - Carrie-Anne Moss (0)
    - Joe Pantoliano (0)
* Batman Begins (2005)
    - Christian Bale (0)
    - Michael Caine (0)
    - Gary Oldman (0)
    - Liam Neeson (0)
    - Katie Holmes (0)
* The Prestige (2006)
    - Christian Bale (0)
    - Michael Caine (0)
    - Hugh Jackman (0)
    - Scarlett Johansson (0)
* The Dark Knight (2008)
    - Christian Bale (0)
    - Michael Caine (0)
    - Gary Oldman (0)
    - Heath Ledger (0)
    - Aaron Eckhart (0)
* Inception (2010)
    - Tom Hardy (0)
    - Joseph Gordon-Levitt (0)
    - Leonardo DiCaprio (0)
    - Elliot Page (0)
    - Ken Watanabe (0)
* The Dark Knight Rises (2012)
    - Christian Bale (0)
    - Gary Oldman (0)
    - Tom Hardy (0)
    - Joseph Gordon-Levitt (0)
    - Anne Hathaway (0)
* Interstellar (2014)
    - Michael Caine (0)
    - Anne Hathaway (0)
    - Matt Damon (0)
    - Matthew McConaughey (0)
    - Jessica Chastain (0)
* Dunkirk (2017)
    - Tom Hardy (0)
    - Cillian Murphy (0)

In [112]:
reset_queries()
nolan = Person.objects.get(name='Christopher Nolan')
qs_other_director = Play.objects.select_related('movie__director') \
                            .exclude(movie__director=nolan) 

qs_other_plays = Play.objects.select_related('actor') \
                .prefetch_related(
                    Prefetch(
                        'actor__plays',
                        queryset=qs_other_director,
                        to_attr='other_plays'
                    )
                )
qs = Movie.objects.filter(director=nolan) \
    .prefetch_related(
        Prefetch(
            'plays',
            queryset=qs_other_plays
        )
    )
for o_movie in qs:
    print(f' * {o_movie.title}')
    for o_play in o_movie.plays.all():
        print(f'    - {o_play.actor.name}')
        for o_play_other in o_play.actor.other_plays:
            print(f'        ~ {o_play_other.movie.director.name}')
            # TODO: compter les real en les dedoublonnant en Python
print(len(connection.queries), connection.queries, sep=': ')

 * Memento
    - Guy Pearce
    - Carrie-Anne Moss
    - Joe Pantoliano
 * Batman Begins
    - Christian Bale
        ~ Mary Harron
    - Michael Caine
    - Liam Neeson
    - Gary Oldman
    - Katie Holmes
 * The Prestige
    - Christian Bale
        ~ Mary Harron
    - Hugh Jackman
    - Scarlett Johansson
    - Michael Caine
 * The Dark Knight
    - Christian Bale
        ~ Mary Harron
    - Heath Ledger
    - Aaron Eckhart
    - Michael Caine
    - Gary Oldman
 * Inception
    - Leonardo DiCaprio
        ~ Martin Scorsese
        ~ Martin Scorsese
        ~ James Cameron
        ~ Steven Spielberg
        ~ Quentin Tarantino
        ~ Alejandro G. Iñárritu
    - Joseph Gordon-Levitt
    - Elliot Page
    - Tom Hardy
        ~ Alejandro G. Iñárritu
    - Ken Watanabe
 * The Dark Knight Rises
    - Christian Bale
        ~ Mary Harron
    - Tom Hardy
        ~ Alejandro G. Iñárritu
    - Anne Hathaway
    - Joseph Gordon-Levitt
    - Gary Oldman
 * Interstellar
    - Matthew McConaug

In [118]:
reset_queries()
qs = Person.objects.filter(birthdate__year=1930) \
        .annotate(
            birth_year=Extract('birthdate', 'year')
        )
for o_person in qs:
    print(f'{o_person.name}, {o_person.birthdate}, {o_person.birth_year}')
connection.queries

Sean Connery, 1930-08-25, 1930
Clint Eastwood, 1930-05-31, 1930
Gene Hackman, 1930-01-30, 1930
Richard Harris, 1930-10-01, 1930
Steve McQueen, 1930-03-24, 1930


[{'sql': 'SELECT "app_people_person"."id", "app_people_person"."name", "app_people_person"."birthdate", "app_people_person"."is_alive", django_date_extract(\'year\', "app_people_person"."birthdate") AS "birth_year" FROM "app_people_person" WHERE "app_people_person"."birthdate" BETWEEN \'1930-01-01\' AND \'1930-12-31\'',
  'time': '0.000'}]

In [127]:
qs = Movie.objects \
    .filter(year=2008) \
    .annotate(
        duration2=Coalesce(
            'duration', 
            0
        ),
        synopsis2=Coalesce(
            'synopsis',
            Value('pas de synopsis', output_field=TextField())
        ))
for o_movie in qs:
    print(f'{o_movie.title} : {o_movie.duration2}, {o_movie.synopsis2}') 

The Dark Knight : 152, pas de synopsis
Quantum of Solace : 106, pas de synopsis
Gran Torino : 116, pas de synopsis


In [87]:
# SQL : groupby

In [ ]:
# projection sans entités
# values/value_list vs only/defer



# Selection de champs
- values/value_list : finaliseur SQL => Python (list[dict], list[tuple])
- only/defer :  QuerySet (Lazy) + mode model

In [136]:
reset_queries()
qs = Movie.objects \
    .values('year') \
    .annotate(nb_film=Count('id'))
print(list(qs))
connection.queries

[{'year': 1962, 'nb_film': 1}, {'year': 1963, 'nb_film': 1}, {'year': 1966, 'nb_film': 1}, {'year': 1968, 'nb_film': 1}, {'year': 1969, 'nb_film': 1}, {'year': 1971, 'nb_film': 1}, {'year': 1977, 'nb_film': 1}, {'year': 1987, 'nb_film': 1}, {'year': 1992, 'nb_film': 1}, {'year': 1995, 'nb_film': 1}, {'year': 1997, 'nb_film': 1}, {'year': 2000, 'nb_film': 2}, {'year': 2002, 'nb_film': 1}, {'year': 2004, 'nb_film': 1}, {'year': 2005, 'nb_film': 1}, {'year': 2006, 'nb_film': 2}, {'year': 2008, 'nb_film': 3}, {'year': 2010, 'nb_film': 1}, {'year': 2011, 'nb_film': 1}, {'year': 2012, 'nb_film': 3}, {'year': 2013, 'nb_film': 1}, {'year': 2014, 'nb_film': 1}, {'year': 2015, 'nb_film': 2}, {'year': 2017, 'nb_film': 1}, {'year': 2018, 'nb_film': 1}, {'year': 2019, 'nb_film': 1}, {'year': 2020, 'nb_film': 1}, {'year': 2021, 'nb_film': 1}, {'year': 2022, 'nb_film': 1}, {'year': 2023, 'nb_film': 1}, {'year': 2024, 'nb_film': 1}, {'year': 2025, 'nb_film': 1}]


[{'sql': 'SELECT "app_movie_movie"."year" AS "year", COUNT("app_movie_movie"."id") AS "nb_film" FROM "app_movie_movie" GROUP BY 1',
  'time': '0.000'}]

In [137]:
reset_queries()
qs = Movie.objects \
    .annotate(decade=F('year') / 10) \
    .values('decade') \
    .annotate(nb_film=Count('id'))
print(list(qs))
connection.queries

[{'decade': 196, 'nb_film': 5}, {'decade': 197, 'nb_film': 2}, {'decade': 198, 'nb_film': 1}, {'decade': 199, 'nb_film': 3}, {'decade': 200, 'nb_film': 10}, {'decade': 201, 'nb_film': 12}, {'decade': 202, 'nb_film': 6}]


[{'sql': 'SELECT ("app_movie_movie"."year" / 10) AS "decade", COUNT("app_movie_movie"."id") AS "nb_film" FROM "app_movie_movie" GROUP BY 1',
  'time': '0.001'}]

In [138]:
reset_queries()
qs = Movie.objects.only('title', 'year') \
    .filter(year = 2015)
print(list(qs))
connection.queries




[<Movie: Movie object (284)>, <Movie: Movie object (296)>]


[{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year" FROM "app_movie_movie" WHERE "app_movie_movie"."year" = 2015',
  'time': '0.000'}]

In [139]:
reset_queries()
qs = Movie.objects.defer('synopsis') \
    .filter(year = 2015)
print(list(qs))
connection.queries

[<Movie: Movie object (284)>, <Movie: Movie object (296)>]


[{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year", "app_movie_movie"."duration", "app_movie_movie"."director_id" FROM "app_movie_movie" WHERE "app_movie_movie"."year" = 2015',
  'time': '0.000'}]

In [141]:
reset_queries()
qs = Movie.objects.defer('synopsis') \
    .filter(year__gt = 2000)
print(qs[0]) # LIMIT 1
connection.queries

Movie object (272)


[{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year", "app_movie_movie"."duration", "app_movie_movie"."director_id" FROM "app_movie_movie" WHERE "app_movie_movie"."year" > 2000 LIMIT 1',
  'time': '0.001'}]

In [142]:
reset_queries()
qs = Movie.objects.defer('synopsis') \
    .filter(year__gt = 2000)
print(qs[:10])  # LIMIT 10
connection.queries

<QuerySet [<Movie: Movie object (272)>, <Movie: Movie object (273)>, <Movie: Movie object (274)>, <Movie: Movie object (275)>, <Movie: Movie object (276)>, <Movie: Movie object (277)>, <Movie: Movie object (278)>, <Movie: Movie object (279)>, <Movie: Movie object (280)>, <Movie: Movie object (281)>]>


[{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year", "app_movie_movie"."duration", "app_movie_movie"."director_id" FROM "app_movie_movie" WHERE "app_movie_movie"."year" > 2000 LIMIT 10',
  'time': '0.000'}]

In [143]:
reset_queries()
qs = Movie.objects.defer('synopsis') \
    .filter(year__gt = 2000)
print(qs[10:20])  # LIMIT 10 OFFSET 10
connection.queries

<QuerySet [<Movie: Movie object (282)>, <Movie: Movie object (283)>, <Movie: Movie object (284)>, <Movie: Movie object (285)>, <Movie: Movie object (291)>, <Movie: Movie object (292)>, <Movie: Movie object (294)>, <Movie: Movie object (295)>, <Movie: Movie object (296)>, <Movie: Movie object (297)>]>


[{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year", "app_movie_movie"."duration", "app_movie_movie"."director_id" FROM "app_movie_movie" WHERE "app_movie_movie"."year" > 2000 LIMIT 10 OFFSET 10',
  'time': '0.000'}]

In [144]:
reset_queries()
qs = Movie.objects.defer('synopsis') \
    .filter(year__gt = 2000)
print(qs.exists()) 
connection.queries

True


[{'sql': 'SELECT 1 AS "a" FROM "app_movie_movie" WHERE "app_movie_movie"."year" > 2000 LIMIT 1',
  'time': '0.000'}]

In [145]:
reset_queries()
qs = Movie.objects.defer('synopsis') \
    .filter(year__gt = 3000)
print(qs.exists())  
connection.queries

False


[{'sql': 'SELECT 1 AS "a" FROM "app_movie_movie" WHERE "app_movie_movie"."year" > 3000 LIMIT 1',
  'time': '0.000'}]

In [146]:
# Moins efficace : ttes les colonnes et ttes les lignes
reset_queries()
qs = Movie.objects.defer('synopsis') \
    .filter(year__gt = 2000)
print(bool(qs))  
connection.queries

True


[{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year", "app_movie_movie"."duration", "app_movie_movie"."director_id" FROM "app_movie_movie" WHERE "app_movie_movie"."year" > 2000',
  'time': '0.000'}]

In [147]:
# Si faux, on reste "efficace"
reset_queries()
qs = Movie.objects.defer('synopsis') \
    .filter(year__gt = 3000)
print(bool(qs))  
connection.queries

False


[{'sql': 'SELECT "app_movie_movie"."id", "app_movie_movie"."title", "app_movie_movie"."year", "app_movie_movie"."duration", "app_movie_movie"."director_id" FROM "app_movie_movie" WHERE "app_movie_movie"."year" > 3000',
  'time': '0.000'}]